# 00.1 — Welcome

Welcome to the `neural_data_decoding` curriculum. By the end of these notebooks you will be able to:

* Open this codebase, navigate it, modify it, and add new architectures, losses, or curriculum schedules.
* Read the MATLAB sources (`Processing_Functions_cgg/`) and the Python sources (`src/neural_data_decoding/`) side-by-side and understand exactly how they map.
* Run a training sweep on real or synthetic neural data, get sensible results, and aggregate them with the existing MATLAB analysis tools.
* Debug training failures: NaN losses, OOM, silent parity drift, deadlocked data loaders.

This notebook is the start of the curriculum. It has **no prerequisites** other than "you have used MATLAB before."

## Section 1 — What MATLAB does

In MATLAB you write `.m` files, run them in the desktop, and the workspace persists between commands. You don't "install" MATLAB packages — everything lives on the MATLAB path. The `Processing_Functions_cgg/` directory works exactly that way.

When you ran `cgg_runAutoEncoder(...)` in MATLAB it would:

1. Build a `cfg` struct of every parameter.
2. Call `cgg_loadDataArray` and `cgg_loadTargetArray` for each trial via `fileDatastore`.
3. Build a `dlnetwork` from the architecture spec.
4. Loop `for kidx = 1:NumEpochs` and call `dlfeval(@modelGradients, ...)`.
5. Save the final weights and a `CM_Table.mat`.

Every step has a Python equivalent in this codebase. The curriculum walks through each of them in order.

## Section 2 — How to actually run this notebook

Before you can execute any code cell below you need a **Jupyter kernel** running this notebook. A kernel is just a Python process that the notebook talks to — it holds your variables, runs your code, and stays alive between cells.

> **Want the deep dive?** Notebook **[00.4 IDE deep dive](00.4_ide_deep_dive.ipynb)** is the foolproof, step-by-step setup guide for VS Code, JupyterLab, and Classic Jupyter, plus a universal troubleshooting flowchart for when something doesn't work. The summary below covers the common case; jump to 00.4 if you hit anything that doesn't behave as described.

You have three good options for getting a kernel attached. Pick whichever matches your editor preference; none is more "correct" than the others.

### Option A — VS Code (probably what you want)

If you already use VS Code for editing code, this is the smoothest path because you keep one editor for both notebooks and the underlying `.py` source.

1. Install the [Python](https://marketplace.visualstudio.com/items?itemName=ms-python.python) and [Jupyter](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) extensions (search them in the Extensions sidebar).
2. Open the repo folder via `File → Open Folder → neural_data_decoding/`.
3. Click this `.ipynb` file in the file tree.
4. In the top-right of the notebook view click **Select Kernel → Python Environments → `.venv/bin/python`**. VS Code remembers the choice per workspace.
5. Click on any code cell and press `Shift+Enter` to run it.

VS Code shows variable values inline, has a built-in debugger (click in the gutter to set breakpoints in code cells), and lets you jump from a function call to its source in `src/neural_data_decoding/` with `F12` / `Cmd+Click`.

### Option B — JupyterLab in your browser

The classic experience. Best if you don't already have an editor preference or want the lab-style sidebar with file browser, kernel manager, and multi-tab layout.

```bash
# From the repo root, with the venv activated:
source .venv/bin/activate
pip install jupyterlab
jupyter lab notebooks/
```

This opens `http://localhost:8888/lab` in your browser. The kernel is selected automatically (it uses the venv's Python because that's what launched it). Click into `00_orientation/00.1_welcome.ipynb`.

### Option C — Classic Jupyter Notebook

The pre-2018 UI. Same kernel mechanics, simpler interface. Use if JupyterLab feels overwhelming.

```bash
source .venv/bin/activate
pip install notebook
jupyter notebook notebooks/
```

### Other ways people interact with this codebase

* **PyCharm Professional** — has built-in Jupyter support similar to VS Code's. The Community Edition does NOT include notebook support; only Professional does.
* **Spyder** — closest in feel to MATLAB's IDE (variable explorer, plot pane, console). Doesn't natively run notebooks but you can use its IPython console to run code straight from `.py` scripts.
* **Plain terminal + your favorite text editor** — for non-notebook work (editing `.py` files in `src/`, running `python -m pytest`, etc.) any editor works. Notebooks specifically need one of the three options above.
* **GitHub web view** — every `.ipynb` renders directly on GitHub (read-only). Useful for sharing a notebook output without anyone needing to run it.

### Picking the right kernel

The single most common gotcha: your editor opens the notebook with the WRONG Python (e.g. the system Python instead of the venv's). Symptoms: `ModuleNotFoundError: neural_data_decoding` even though you ran `pip install -e .`.

To diagnose, run this in any code cell once your kernel starts:

```python
import sys
print(sys.executable)
```

The path should end in `.venv/bin/python` (or `.venv\Scripts\python.exe` on Windows). If it doesn't, switch kernels — see the per-editor instructions above.


## Section 3 — The Python concepts you need

(This section assumes you've attached a kernel via Section 2. Try running the first code cell below — if it errors, double-check Section 2.)

### Jupyter notebooks (this thing you're reading)

A notebook is a sequence of **cells**. Each cell is either:

* **Markdown** — explanatory text like this one. Double-click to edit; press `Shift+Enter` to render.
* **Code** — Python that runs in a persistent **kernel**. Press `Shift+Enter` to execute.

**The kernel is the workspace.** Variables you create in one code cell are available in every cell after it. If you restart the kernel (Kernel → Restart in the toolbar) you lose all variables and have to re-run the cells you care about.

Try it — run the cell below by clicking on it and pressing `Shift+Enter`:


In [ ]:
x = 1 + 2
x

The last expression in a code cell is echoed automatically (just like MATLAB without a semicolon).

In [ ]:
y = x * 10
print(f"x = {x}, y = {y}")

Notice three things:

1. `x` was defined in the previous cell and is still in scope here.
2. `print(...)` is used for explicit output; bare `y` would also echo (try it).
3. `f"x = {x}"` is an **f-string** — Python's string-interpolation syntax. The MATLAB equivalent is `sprintf('x = %d', x)`.

## Section 4 — The `neural_data_decoding` package

The production code lives in `../src/neural_data_decoding/`. When you installed the project with `pip install -e .` (see notebook 00.2), Python made it importable from anywhere — including from this notebook.

In [ ]:
import neural_data_decoding
print(f"Version: {neural_data_decoding.__version__}")

If that cell errored with `ModuleNotFoundError`, jump to **notebook 00.2** — your environment isn't set up yet.

Let's verify the curriculum can reach the main components:

In [ ]:
from neural_data_decoding.data import MatFileTrialDataset, SyntheticTrialDataset
from neural_data_decoding.sweeps import total_sweep_count, lookup
from neural_data_decoding.interop import build_matlab_run_dirs

print(f"Sweep entries available: {total_sweep_count()}")
print(f"Sweep entry 1: {lookup(1).description}")
print(f"MatFileTrialDataset: {MatFileTrialDataset.__module__}")
print(f"SyntheticTrialDataset: {SyntheticTrialDataset.__module__}")

Each of those symbols has a dedicated notebook somewhere in the curriculum that explains what it is, what its MATLAB analog is, and how to use it.

## Section 5 — How the curriculum is organized

There are **10 modules**, each in a numbered subdirectory:

| Module | Topic | Companion code milestone |
|---|---|---|
| 00 | Orientation (you are here) | — |
| 01 | Python for MATLAB users | — |
| 02 | NumPy & PyTorch basics | Milestone 0 ("is the harness alive") |
| 03 | Data pipeline | Milestone 0 + A ("tracer bullet") |
| 04 | Architecture | Milestone B (encoder + classifier) |
| 05 | Training loop | Milestone B/C (training mechanics) |
| 06 | Loss orchestration | Milestone C (Optimal-config losses) |
| 07 | Dynamic curriculum | Milestone C (schedules) |
| 08 | Output & analysis | Milestone C/D (interop) |
| 09 | Production deployment | Milestone D (cluster + sweeps) |

Inside each module, notebooks are numbered `MM.N_topic.ipynb`. You can read them in order or jump around — the [curriculum map](../README.md) shows the prerequisite graph.

### Every notebook follows the same 6-section template

1. **What MATLAB does** — the actual `cgg_*` MATLAB code in plain English.
2. **The Python concept(s) you need** — first principles, with micro-examples.
3. **The `neural_data_decoding` implementation** — production code, line-by-line annotated.
4. **Hands-on exercises** — small problems with hidden solutions.
5. **Diagnostic / debugging walkthrough** — common MATLAB→Python errors.
6. **Further reading** — PyTorch docs + relevant repo docs.

## Section 6 — Hands-on exercise

Find one MATLAB function in `Processing_Functions_cgg/` that you understand well from your existing work. Then check which curriculum notebook teaches its Python equivalent.

(There's no automated check — this exercise is to get you oriented to the cross-reference style. The [curriculum map](../README.md) is your friend.)

## Section 7 — Common errors (MATLAB → Python pitfalls)

### `ModuleNotFoundError: No module named 'neural_data_decoding'`

Your kernel doesn't have the project installed, OR the notebook attached to the wrong Python interpreter. Run `import sys; print(sys.executable)` in a code cell — if it doesn't end in `.venv/bin/python`, switch kernels (Section 2 has the per-editor instructions). If the kernel IS the venv's Python, run `pip install -e .` from the repo root.

### "Select Kernel" greys out / no kernel options appear

The editor doesn't know about your venv yet. In VS Code: open the command palette (`Cmd/Ctrl+Shift+P`), run **Python: Select Interpreter**, point at `.venv/bin/python`. In JupyterLab: this typically means JupyterLab wasn't launched from an activated venv — `deactivate`, re-`source .venv/bin/activate`, and re-launch `jupyter lab`.

### A cell shows `<IPython.core.display.Image object at 0x…>` instead of an image

You forgot to wrap with `display(...)` or are missing a Jupyter dependency. Restart the kernel; run all cells in order.

### "My variable from a few cells back is gone!"

The kernel was restarted. Run the cells above the one that broke. The toolbar's **Restart & Run All** button is your friend.

### A cell hangs forever

Look for an `In [*]:` marker on the left — that's the kernel still working. Press the square **Interrupt** button and try a smaller input.

### "In MATLAB this would just work"

Python is stricter about types, scopes, and namespaces than MATLAB. The error messages are detailed — read the full traceback bottom-to-top (the bottom line is the immediate error; the lines above show how the call got there). **Notebook 01.6** walks through reading tracebacks in detail.


## Section 8 — Further reading

* [`notebooks/README.md`](../README.md) — the full curriculum map and "I'm coming from X background" guide.
* [`CLAUDE.md`](../../CLAUDE.md) — repo bootstrap (CWD conventions, where the venv lives).
* [`HANDOFF.md`](../../HANDOFF.md) — current development state and what each milestone delivered.
* [`docs/PLAN.md`](../../docs/PLAN.md) — the original migration plan with every Critical Note.
* [Jupyter notebook docs](https://jupyter-notebook.readthedocs.io/) — the canonical reference.

When you're ready, head to **[00.2 Set up your environment](00.2_set_up_your_environment.ipynb)**.